## install libraries

In [17]:
pip install mediapipe opencv-python requests

Note: you may need to restart the kernel to use updated packages.


## install AI model files

In [18]:
import urllib.request
import os

# The model file URL (hosted by Google)
MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task'

MODEL_PATH = 'pose_landmarker.task'

# Only download if we don't already have it
if not os.path.exists(MODEL_PATH):
    print(' Downloading AI model... (about 5MB, takes a few seconds)')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(' Model downloaded!')
else:
    print(' Model already exists, skipping download.')

print(f' Model saved at: {os.path.abspath(MODEL_PATH)}')

 Model already exists, skipping download.
 Model saved at: d:\mediapipe_pose_estimation\pose_landmarker.task


In [19]:
import cv2
import numpy as np
import mediapipe as mp

# Import from the NEW mediapipe tasks API
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions
from mediapipe.tasks.python.core import base_options as mp_base

# This tells mediapipe to process video frame by frame (not a static image)
VisionRunningMode = mp_vision.RunningMode

# Verify everything works
print(' cv2 version      :', cv2.__version__)
print(' mediapipe version:', mp.__version__)
print(' numpy version    :', np.__version__)
print('\n All imports successful!')

 cv2 version      : 4.13.0
 mediapipe version: 0.10.32
 numpy version    : 2.4.2

 All imports successful!


## STEP 3 — Helper Functions

We write two helper functions we'll use throughout:

### Function 1: `calculate_angle(a, b, c)`
Measures the angle at joint `b` using 3 body points.
```
  SHOULDER (a)
      \
       \  ← angle here at elbow
        ELBOW (b)
       /
      /
  WRIST (c)
```

### Function 2: `draw_landmarks_on_frame(frame, detection_result)`
Draws the skeleton (dots + connecting lines) on the video frame.

In [20]:
def calculate_angle(a, b, c):

    a = np.array(a)  #  shoulder
    b = np.array(b)  #  elbow angle measured HERE
    c = np.array(c)  #  wrist

    # arctan2 gives the angle of each line from horizontal
    # Subtracting them gives the angle between the two lines at point b
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    # Fix angles > 180 (arctan2 can go negative)
    if angle > 180.0:
        angle = 360 - angle

    return angle

In [21]:
# ── Landmark index constants (the 33 body points) ──
# These numbers come from MediaPipe's documentation
LEFT_SHOULDER = 11
LEFT_ELBOW    = 13
LEFT_WRIST    = 15
RIGHT_SHOULDER = 12
RIGHT_ELBOW    = 14
RIGHT_WRIST    = 16

# Skeleton connections — which dots to connect with lines
# Each tuple = (start_point_index, end_point_index)
POSE_CONNECTIONS = [
    (11,13),(13,15),  # left arm
    (12,14),(14,16),  # right arm
    (11,12),          # shoulders
    (11,23),(12,24),  # torso sides
    (23,24),          # hips
    (23,25),(25,27),  # left leg
    (24,26),(26,28),  # right leg
]

In [22]:
def draw_skeleton(frame, landmarks, h, w):

    # Draw connecting LINES (the skeleton bones)
    for (start_idx, end_idx) in POSE_CONNECTIONS:
        if start_idx < len(landmarks) and end_idx < len(landmarks):
            # Landmark x,y are 0.0-1.0 fractions, multiply by w/h for pixels
            start = (int(landmarks[start_idx].x * w), int(landmarks[start_idx].y * h))
            end   = (int(landmarks[end_idx].x * w),   int(landmarks[end_idx].y * h))
            cv2.line(frame, start, end, (245, 66, 230), 2)  # pink lines

    # Draw DOTS on each landmark
    for lm in landmarks:
        cx = int(lm.x * w)
        cy = int(lm.y * h)
        cv2.circle(frame, (cx, cy), 5, (245, 117, 66), -1)  # orange dots

In [23]:
# Quick angle test 
print('Testing angle function:')
print(f'  Straight arm → {calculate_angle([0,0],[1,0],[2,0]):.1f}° (expected ~180°)')
print(f'  Right angle  → {calculate_angle([0,0],[1,0],[1,1]):.1f}° (expected ~90°)')
print(' All helper functions ready!')

Testing angle function:
  Straight arm → 180.0° (expected ~180°)
  Right angle  → 90.0° (expected ~90°)
 All helper functions ready!


## Test Webcam (No Pose Detection Yet)

In [25]:
# 0 = first webcam. Try 1 if this doesn't show anything.
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print('Cannot open webcam. Try changing 0 to 1 in VideoCapture()')
else:
    print(' Webcam opened! Press Q in the video window to close.')

while cap.isOpened():
    ret, frame = cap.read()   # grab one frame
    if not ret:
        break

    cv2.imshow('Webcam Test - Press Q to close', frame)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(' Webcam closed.')

 Webcam opened! Press Q in the video window to close.
 Webcam closed.


## live pose detection

In [27]:
# Create the pose detector 
# BaseOptions tells it WHERE the AI model file is
base_options = mp_python.BaseOptions(model_asset_path='pose_landmarker.task')

options = PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=VisionRunningMode.VIDEO,  # VIDEO mode = process frame by frame
    min_pose_detection_confidence=0.5,     # 50% confidence to detect a person
    min_pose_presence_confidence=0.5,      # 50% confidence person is present
    min_tracking_confidence=0.5            # 50% confidence to keep tracking
)

cap = cv2.VideoCapture(0)
frame_number = 0  # We need frame timestamps for VIDEO mode

# 'with' statement automatically cleans up the detector when done
with PoseLandmarker.create_from_options(options) as detector:
    print('Pose detector ready! Press Q to close the window.')
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_number += 1
        h, w, _ = frame.shape  # get height and width of frame in pixels

        # ── Convert frame for MediaPipe ──
        # MediaPipe needs its own Image format (not a numpy array)
        # SRGB = standard RGB color format
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # ── Run AI detection ──
        # frame_number * 33 converts frame count to milliseconds (assuming ~30fps)
        result = detector.detect_for_video(mp_image, frame_number * 33)

        # ── Draw skeleton if a person was detected ──
        # result.pose_landmarks is a list of detected people
        # Each person has 33 landmarks
        if result.pose_landmarks:
            # result.pose_landmarks[0] = first detected person's landmarks
            landmarks = result.pose_landmarks[0]
            draw_skeleton(frame, landmarks, h, w)

        cv2.imshow('Pose Detection - Press Q to close', frame)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print('Done!')

Pose detector ready! Press Q to close the window.
Done!
